In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import TextLoader

In [3]:
loader = TextLoader("../data/news.txt",encoding="utf-8")
docs = loader.load()
print(f"총 글자 수: {len(docs[0].page_content)}")
print("\n========= 앞부분 미리보기 ==========\n")
print(docs[0].page_content[:500])

총 글자 수: 1708

========= 앞부분 미리보기 ==========

제목: 
AI2, 상업 활용까지 자유로운 '진짜' 오픈 소스 LLM '올모' 출시

내용:
앨런AI연구소(AI2)가 완전한 오픈 소스 대형언어모델(LLM) '올모(OLMo)’를 출시했다. 데이터 수집, 학습, 배포의 전 과정을 투명하게 공개한 데다 상업적 사용까지 허용한 진정한 의미의 오픈 소스 LLM이라는 평가다.
벤처비트는 1일(현지시간) 비영리 민간 AI 연구기관인 AI2가 ‘최초의 진정한 오픈 소스 LLM 및 프레임워크’라고 소개한 ‘올모’를 출시했다고 보도했다. 
이에 따르면 올모는 모델 코드와 모델 가중치뿐만 아니라 훈련 코드, 훈련 데이터, 관련 툴킷 및 평가 툴킷도 제공한다. 이를 통해 모델이 어떻게 구축되었는지 심층적으로 분석, LLM의 작동 방식과 응답을 생성하는 원리를 더 잘 이해할 수 있다. 
올모 프레임워크는 70억 매개변수의 ‘올모 7B’ 등 4가지 변형 모델과 10억 매개변수의 ‘올모 1B’ 모델을 제공한다. 모델들은 훈련 데이터를 생성하는 코드를 포함해 


In [4]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt(
    "teddynote/summary-stuff-documents-korean",
    dangerously_pull_public_prompt=True,
)
prompt.pretty_print()

Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points in KOREAN.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. Translate the summary into KOREAN if it is written in ENGLISH.
5. DO NOT translate any technical terms.
6. DO NOT include any unnecessary information.

CONTEXT:
{context}

SUMMARY:"



In [5]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_teddynote.callbacks import StreamingCallback

llm = ChatOpenAI(
    model="gpt-4o-mini",
    streaming=True,
    temperature=0,
    callbacks=[StreamingCallback()],
)

stuff_chain = create_stuff_documents_chain(llm, prompt)
answer = stuff_chain.invoke({"context": docs})

- 🚀 앨런AI연구소(AI2)가 완전한 오픈 소스 LLM '올모(OLMo)'를 출시했다.  
- 📊 데이터 수집, 학습, 배포 과정을 투명하게 공개하고 상업적 사용을 허용한다.  
- 🔍 올모는 모델 코드, 가중치, 훈련 코드, 데이터 및 평가 툴킷을 제공한다.  
- 🧩 70억 매개변수의 '올모 7B'와 10억 매개변수의 '올모 1B' 모델을 포함한 4가지 변형 모델이 있다.  
- 📈 훈련 데이터는 3조개의 토큰으로 구성된 AI2의 '돌마(Dolma)' 데이터 세트를 기반으로 한다.  
- 📝 아파치 2.0 라이선스에 따라 상업적 활용에 제한이 없다.  
- 💡 AI2의 올모 프로젝트 책임자는 투명성이 부족한 현재의 언어 모델에 대한 문제를 지적했다.  
- 🏆 올모는 상업용 제품과 동등한 성능을 보여주며, 일부 벤치마크 테스트에서 우수한 결과를 기록했다.  
- 🌍 비영어권 언어와 코드 생성 기능에 대한 제약이 있다.  
- 🔄 AI2는 올모를 지속적으로 개선할 계획이다.  
- 🌐 올모의 모든 리소스는 깃허브 및 허깅페이스에서 무료로 제공된다.  

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
docs = docs[3:8]
print(f"총 페이지수: {len(docs)}")

총 페이지수: 5


In [7]:
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

map_prompt = client.pull_prompt(
    "teddynote/map-prompt",
    dangerously_pull_public_prompt=True
)

map_prompt.pretty_print()

================================ System Message ================================

You are a professional main thesis extractor.

================================ Human Message =================================

Your task is to extract main thesis from given documents. Answer should be in same language as given document. 

#Format: 
- thesis 1
- thesis 2
- thesis 3
- ...

Here is a given document: 
{doc}

Write 1~5 sentences.
#Answer:


In [8]:
map_chain = map_prompt | llm | StrOutputParser()

In [9]:
doc_summaries = map_chain.batch(docs)

In [10]:
print(doc_summaries[0])

- 미국 바이든 대통령이 안전하고 신뢰할 수 있는 AI 개발과 사용을 위한 행정명령을 발표하였다.
- 이 행정명령은 AI의 안전과 보안 기준, 개인정보 보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제협력을 포함한다.
- AI 시스템 개발 기업은 안전 테스트 결과와 시스템 정보를 미국 정부와 공유해야 하며, AI의 책임 있는 사용을 위한 모범사례를 개발해야 한다.
- 소비자 보호와 근로자 지원을 위해 의료 분야에서의 책임 있는 AI 사용을 촉진하고, AI로 인한 근로자 피해를 완화하는 원칙을 마련할 예정이다.
- 국가AI연구자원(NAIRR)을 통해 AI 연구를 촉진하고, 외국인 전문가들이 미국에서 공부하고 취업할 수 있도록 지원할 계획이다.


In [11]:
reduce_prompt = client.pull_prompt(
    "teddynote/reduce-prompt",
    dangerously_pull_public_prompt=True,
)

reduce_prompt.pretty_print()

================================ System Message ================================

You are a professional summarizer. You are given a list of summaries of documents and you are asked to create a single summary of the documents.

================================ Human Message =================================

#Instructions: 
1. Extract main points from a list of summaries of documents
2. Make final summaries in bullet points format.
3. Answer should be written in {language}.

#Format: 
- summary 1
- summary 2
- summary 3
- ...

Here is a list of summaries of documents: 
{doc_summaries}

#SUMMARY:


In [12]:
reduce_chain = reduce_prompt | llm | StrOutputParser()

In [13]:
from langchain_teddynote.messages import stream_response

answer = reduce_chain.stream(
    {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
)

stream_response(answer)

- 미국 바이든 대통령이 AI 개발과 사용을 위한 행정명령을 발표하였다.
- 행정명령은 AI의 안전, 개인정보 보호, 형평성, 소비자 보호, 노동자 지원, 혁신 촉진, 국제협력을 포함한다.
- AI 시스템 개발 기업은 안전 테스트 결과를 정부와 공유하고, 책임 있는 사용을 위한 모범사례를 개발해야 한다.
- 의료 분야에서의 AI 사용을 촉진하고 근로자 피해를 완화하는 원칙을 마련할 예정이다.
- G7은 '히로시마 AI 프로세스'를 통해 AI 기업을 위한 국제 행동강령에 합의하였다.
- 행동강령은 AI 수명주기 전반의 위험 평가, 투명성 보장, 정보공유를 요구한다.
- AI 생성 콘텐츠의 신뢰성을 높이기 위한 인증 및 출처 확인 메커니즘 개발이 제안되었다.
- 28개국이 AI 안전성 정상회의에서 AI 안전 보장을 위한 블레츨리 선언을 발표하였다.
- 영국 총리는 AI 안전 연구소 출범과 안전성 시험 계획 수립을 발표하였다.
- 미국 법원은 생성 AI 기업에 대한 저작권 침해 소송을 기각하였으나, 일부 소송은 계속 진행된다.
- FTC는 생성 AI의 소비자 피해 가능성과 빅테크의 시장 지배력 우려를 표명하며, 법적 권한을 활용해 AI 관련 불법 행위에 대처하겠다고 밝혔다.

In [14]:
from langchain_core.runnables import chain

@chain
def map_reduce_chain(docs):
    map_llm = ChatOpenAI(
        temperature=0, model="gpt-4o-mini"
    )

    map_prompt = client.pull_prompt(
        "teddynote/map-prompt",
        dangerously_pull_public_prompt=True,
    )

    map_chain = map_prompt | map_llm | StrOutputParser()

    doc_summaries = map_chain.batch(docs)

    reduce_prompt = client.pull_prompt(
        "teddynote/reduce-prompt",
        dangerously_pull_public_prompt=True,
    )

    reduce_llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        callbacks=[StreamingCallback()],
        streaming=True,
    )

    reduce_chain = reduce_prompt | reduce_llm | StrOutputParser()

    return reduce_chain.invoke(
        {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
    )

In [15]:
answer = map_reduce_chain.invoke(docs)

- 미국 바이든 대통령이 AI 개발과 사용을 위한 행정명령을 발표하였다.
- 행정명령은 AI의 안전, 개인정보 보호, 형평성, 소비자 보호, 노동자 지원, 혁신 촉진, 국제 협력을 포함한다.
- AI 시스템 개발 기업은 안전 테스트 결과를 정부와 공유하고, 책임 있는 사용을 위한 모범사례를 개발해야 한다.
- 의료 분야에서 AI 사용을 촉진하고 근로자 피해를 완화하는 원칙을 마련할 예정이다.
- G7은 '히로시마 AI 프로세스'를 통해 AI 기업을 위한 국제 행동강령에 합의하였다.
- 행동강령은 AI 수명주기 전반의 위험 평가, 투명성 보장, 정보공유를 요구한다.
- G7은 AI 생성 콘텐츠의 신뢰성을 높이기 위한 인증 및 출처 확인 메커니즘 개발을 제안하였다.
- 28개국이 AI 안전성 정상회의에서 AI 안전 보장을 위한 블레츨리 선언을 발표하였다.
- 선언은 AI 시스템의 안전을 보장하기 위한 협력의 중요성을 강조하였다.
- 미국 캘리포니아 법원은 생성 AI 기업에 대한 저작권 침해 소송을 기각하였다.
- FTC는 생성 AI로 인한 소비자와 창작자의 피해 가능성을 경고하며, 시장 지배력 강화 우려를 표명하였다.
- FTC는 소비자 보호와 공정한 경쟁 시장 유지를 위해 모든 권한을 활용하겠다고 강조하였다.